# DPA - Volume (Part 1): Network Traffic Analysis

Our research questions:

1. What traffic patterns distinguish normal activity from attacks?
2. Which network services and traffic characteristics are associated with different attack categories?

We have four steps so far:

1. Check the source files and small samples.
2. Define the column names and types.
3. Test reading CSV and saving Parquet with a small sample.
4. Read all the data, clean it, and save the row counts.

Run the cells from top to bottom. Keep `DOWNLOAD_DATA = False` if you already have the data. Step 1 uses Python only. The next steps need PySpark, and Step 3 also needs Java. Our Docker setup includes both.


## Optional - Download the data

Part 1 uses the original traffic files in `data/CSECICIDS2018_original/` and the IANA registry. The traffic files are about 6.89 GB.

Set `DOWNLOAD_DATA = True` to download them. Existing nonempty files are skipped. If a download stops, rerun this cell; that file starts again. Downloading needs Python only.


In [1]:
DOWNLOAD_DATA = False  # Set to True if you need to download the data.

from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif (ROOT / "DPA-Volume-Part1-Project" / "downloaders").is_dir():
    ROOT = ROOT / "DPA-Volume-Part1-Project"
if not (ROOT / "downloaders" / "download_data.py").is_file():
    raise FileNotFoundError("Open the notebook from the project folder or notebooks/.")
print("Project folder:", ROOT)

if DOWNLOAD_DATA:
    import runpy
    downloader = runpy.run_path(str(ROOT / "downloaders" / "download_data.py"))
    downloader["prepare_iana"](ROOT / "data")
    downloader["prepare_traffic"](ROOT / "data")
    print("Data is ready.")
else:
    print("Download skipped.")


Project folder: /home/jovyan/work
Download skipped.


## Step 1 - Check the source files

We check the columns and some values before choosing the types and cleaning rules.

For traffic, we read up to 100 rows at five positions in each file: the start, 25%, 50%, 75%, and 99%. These small samples help us check the data. They are not random samples, so we cannot use them to estimate the number of attacks in the full dataset.

IANA is small, so we check the whole file. Nothing is cleaned in this step.


In [2]:
import csv
import io
import json
from collections import Counter
from datetime import datetime
from itertools import islice

traffic_folder = ROOT / "data" / "CSECICIDS2018_original"
traffic_files = sorted(traffic_folder.glob("*.csv"))
iana_folder = ROOT / "data" / "Service and port name"
iana_files = sorted(iana_folder.glob("service-names-port-numbers*.csv"))
assert traffic_files, f"No traffic CSVs found in {traffic_folder}"
assert iana_files, f"No IANA registry found in {iana_folder}"
# Prefer the standard filename, while accepting the existing '(3)' filename.
registry_file = next((p for p in iana_files if p.name == "service-names-port-numbers.csv"), iana_files[0])

print("Traffic CSVs:", len(traffic_files))
for path in traffic_files:
    print(f"  {path.name}: {path.stat().st_size / 1024**3:.2f} GiB")
print("IANA:", registry_file.name)

Traffic CSVs: 10
  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 0.33 GiB
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 0.31 GiB
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 0.36 GiB
  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 3.78 GiB
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 0.10 GiB
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 0.35 GiB
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 0.36 GiB
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 0.33 GiB
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 0.31 GiB
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 0.19 GiB
IANA: service-names-port-numbers (3).csv


### 1A - Check the headers

We check column names, their order, and repeated names. Nine files have 80 columns. The 20 February file has 84, with four extra columns: `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`. We will define a schema for each layout.


In [3]:
traffic_headers = {}
header_summary = []
for path in traffic_files:
    with path.open(encoding="utf-8-sig", newline="") as source:
        header = next(csv.reader(source))
    assert len(header) == len(set(header)), f"Duplicate column names in {path.name}"
    traffic_headers[path.name] = header
    reference_header = traffic_headers[traffic_files[0].name]
    header_summary.append({
        "file": path.name,
        "size_bytes": path.stat().st_size,
        "column_count": len(header),
        "matches_first_header": header == reference_header,
    })
    print(f"{path.name}: {len(header)} columns; same names/order = {header == reference_header}")

print("\nColumns in the first traffic file:")
for position, name in enumerate(reference_header, start=1):
    print(f"{position:2}. {name}")

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True


Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 84 columns; same names/order = False
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True


Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True

Columns in the first traffic file:
 1. Dst Port
 2. Protocol
 3. Timestamp
 4. Flow Duration
 5. Tot Fwd Pkts
 6. Tot Bwd Pkts
 7. TotLen Fwd Pkts
 8. TotLen Bwd Pkts
 9. Fwd Pkt Len Max
10. Fwd Pkt Len Min
11. Fwd Pkt Len Mean
12. Fwd Pkt Len Std
13. Bwd Pkt Len Max
14. Bwd Pkt Len Min
15. Bwd Pkt Len Mean
16. Bwd Pkt Len Std
17. Flow Byts/s
18. Flow Pkts/s
19. Flow IAT Mean
20. Flow IAT Std
21. Flow IAT Max
22. Flow IAT Min
23. Fwd IAT Tot
24. Fwd IAT Mean
25. Fwd IAT Std
26. Fwd IAT Max
27. Fwd IAT Min
28. Bwd IAT Tot
29. Bwd IAT Mean
30. Bwd IAT Std
31. Bwd IAT Max
32. Bwd IAT Min
33. Fwd PSH Flags
34. Bwd PSH Flags
35. Fwd URG Flags
36

### 1B - Read small samples

`seek` lets us jump to different parts of a large file. We skip the partial line at that position, then read up to 100 rows. We check that each row has the right number of columns and keep the samples grouped by file.


In [4]:
ROWS_PER_WINDOW = 100
WINDOW_POSITIONS = (0.0, 0.25, 0.5, 0.75, 0.99)

def inspect_window(path, header, fraction):
    with path.open("rb") as source:
        source.seek(int(path.stat().st_size * fraction))
        source.readline()  # Skip the header, or the partial line at this position.
        with io.TextIOWrapper(source, encoding="utf-8", newline="") as text:
            records = list(islice(csv.reader(text), ROWS_PER_WINDOW))
    assert all(len(row) == len(header) for row in records), f"Unexpected CSV record width in {path.name}"
    return [dict(zip(header, row)) for row in records]

sample_rows = []
sample_rows_by_file = {}
window_summary = []
for path in traffic_files:
    sample_rows_by_file[path.name] = []
    for fraction in WINDOW_POSITIONS:
        rows = inspect_window(path, traffic_headers[path.name], fraction)
        sample_rows.extend(rows)
        sample_rows_by_file[path.name].extend(rows)
        window_summary.append({"file": path.name, "position_fraction": fraction, "sampled_rows": len(rows)})
    print(f"{path.name}: inspected {sum(w['sampled_rows'] for w in window_summary if w['file'] == path.name)} records")
print("Total inspected traffic records:", len(sample_rows))

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Total inspected traffic records: 5000


### 1C - Check labels, protocols, dates, and numbers

These counts are for the samples only. We keep the original attack labels. Traffic uses numeric protocol codes, which we will match to IANA's protocol names later.

Dates use day/month/year, sometimes with fractional seconds. We keep them as text for now. A negative TCP window value such as `-1` may be a special value, so we should not treat every negative number as an error.


In [5]:
sample_labels = Counter(row["Label"] for row in sample_rows)
sample_protocols = Counter(row["Protocol"] for row in sample_rows)
print("Labels in inspected windows:", dict(sample_labels))
print("Raw protocol codes:", dict(sample_protocols))

formats = ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M:%S.%f")
format_counts = Counter()
parsed_times = []
unparsed_times = []
for row in sample_rows:
    for timestamp_format in formats:
        try:
            parsed_times.append(datetime.strptime(row["Timestamp"], timestamp_format))
            format_counts[timestamp_format] += 1
            break
        except ValueError:
            continue
    else:
        unparsed_times.append(row["Timestamp"])
print("Timestamp formats matched:", dict(format_counts))
print("Unparsed sampled timestamps:", len(unparsed_times))
if parsed_times:
    print("Sample timestamp range:", min(parsed_times), "to", max(parsed_times))

example_columns = ["Dst Port", "Protocol", "Timestamp", "Flow Duration",
                   "Tot Fwd Pkts", "Tot Bwd Pkts", "Label"]
examples = {}
for row in sample_rows:
    if row["Label"] not in examples:
        examples[row["Label"]] = {column: row[column] for column in example_columns}
print("\nOne raw example per observed label:")
print(json.dumps(examples, indent=2))

sentinel_examples = {}
for column in ("Init Fwd Win Byts", "Init Bwd Win Byts"):
    sentinel_examples[column] = sorted({row[column] for row in sample_rows if row[column].startswith("-")})
print("Observed negative values requiring interpretation:", sentinel_examples)


Labels in inspected windows: {'Benign': 4140, 'DoS attacks-SlowHTTPTest': 103, 'DoS attacks-Hulk': 164, 'DDoS attacks-LOIC-HTTP': 37, 'Infilteration': 144, 'DoS attacks-GoldenEye': 56, 'FTP-BruteForce': 6, 'SSH-Bruteforce': 99, 'DDOS attack-HOIC': 251}
Raw protocol codes: {'6': 3511, '0': 325, '17': 1164}
Timestamp formats matched: {'%d/%m/%Y %H:%M:%S': 5000}
Unparsed sampled timestamps: 0
Sample timestamp range: 2018-02-14 01:22:05 to 2018-03-02 12:57:21

One raw example per observed label:
{
  "Benign": {
    "Dst Port": "443",
    "Protocol": "6",
    "Timestamp": "02/03/2018 08:47:38",
    "Flow Duration": "141385",
    "Tot Fwd Pkts": "9",
    "Tot Bwd Pkts": "7",
    "Label": "Benign"
  },
  "DoS attacks-SlowHTTPTest": {
    "Dst Port": "21",
    "Protocol": "6",
    "Timestamp": "16/02/2018 10:12:14",
    "Flow Duration": "21",
    "Tot Fwd Pkts": "1",
    "Tot Bwd Pkts": "1",
    "Label": "DoS attacks-SlowHTTPTest"
  },
  "DoS attacks-Hulk": {
    "Dst Port": "80",
    "Protoco

### 1D - Check IANA

We check missing service names, ports, and protocols. Some ports are ranges, and several service names can share the same port and protocol. We only count these cases here.

IANA tells us which services are registered for a port. It does not prove which application produced the traffic.


In [6]:
with registry_file.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    registry_header = reader.fieldnames
    registry_rows = list(reader)  # Only the small IANA registry, never the traffic files.
print("IANA columns:", registry_header)
assert len(registry_header) == len(set(registry_header)), "Duplicate IANA column names"
assert {"Port Number", "Transport Protocol", "Service Name"}.issubset(registry_header)
assert all(None not in row and None not in row.values() for row in registry_rows), "Unexpected IANA record width"

registry_keys = Counter((row["Port Number"], row["Transport Protocol"]) for row in registry_rows)
range_ports = [row["Port Number"] for row in registry_rows if "-" in row["Port Number"]]
iana_summary = {
    "file": registry_file.name,
    "row_count": len(registry_rows),
    "column_count": len(registry_header),
    "transport_protocol_counts": dict(Counter(row["Transport Protocol"] for row in registry_rows)),
    "rows_with_port_ranges": len(range_ports),
    "port_range_examples": list(dict.fromkeys(range_ports))[:5],
    "rows_with_blank_port": sum(not row["Port Number"].strip() for row in registry_rows),
    "rows_with_blank_protocol": sum(not row["Transport Protocol"].strip() for row in registry_rows),
    "rows_with_blank_service": sum(not row["Service Name"].strip() for row in registry_rows),
    "repeated_literal_key_count": sum(count > 1 for count in registry_keys.values()),
    "extra_rows_on_repeated_literal_keys": sum(count - 1 for count in registry_keys.values()),
}
print(json.dumps(iana_summary, indent=2))
print("Example registered service:", next(row for row in registry_rows if row["Service Name"] and row["Port Number"].isdigit()))

IANA columns: ['Service Name', 'Port Number', 'Transport Protocol', 'Description', 'Assignee', 'Contact', 'Registration Date', 'Modification Date', 'Reference', 'Service Code', 'Unauthorized Use Reported', 'Assignment Notes']
{
  "file": "service-names-port-numbers (3).csv",
  "row_count": 14536,
  "column_count": 12,
  "transport_protocol_counts": {
    "tcp": 6608,
    "udp": 6357,
    "sctp": 93,
    "dccp": 11,
    "": 1467
  },
  "rows_with_port_ranges": 766,
  "port_range_examples": [
    "225-241",
    "249-255",
    "272-279",
    "288-299",
    "301-307"
  ],
  "rows_with_blank_port": 1073,
  "rows_with_blank_protocol": 1467,
  "rows_with_blank_service": 1724,
  "repeated_literal_key_count": 230,
  "extra_rows_on_repeated_literal_keys": 1300
}
Example registered service: {'Service Name': 'tcpmux', 'Port Number': '1', 'Transport Protocol': 'tcp', 'Description': 'TCP Port Service Multiplexer', 'Assignee': '[Mark_Lottor]', 'Contact': '[Mark_Lottor]', 'Registration Date': '', 'Mod

### 1E - Save what we found

Save the inspection summary as JSON and the header checks as CSV. These files describe the inspection; they are not the final analysis or cleaning results.


In [7]:
inspection_folder = ROOT / "reports" / "inspection" / "original"
inspection_folder.mkdir(parents=True, exist_ok=True)
inspection_summary = {
    "scope": "Source inspection only; no cleaning, typed ingestion, or Spark analysis performed",
    "traffic": {
        "source_folder": str(traffic_folder.relative_to(ROOT)),
        "file_count": len(traffic_files),
        "total_file_bytes": sum(path.stat().st_size for path in traffic_files),
        "header_summary": header_summary,
        "headers_by_file": traffic_headers,
        "sampling_method": "100 records after each byte position: 0%, 25%, 50%, 75%, 99%; not statistically representative",
        "window_summary": window_summary,
        "sample_row_count": len(sample_rows),
        "sample_label_counts": dict(sample_labels),
        "sample_protocol_counts": dict(sample_protocols),
        "sample_timestamp_formats": dict(format_counts),
        "sample_timestamp_min": min(parsed_times).isoformat() if parsed_times else None,
        "sample_timestamp_max": max(parsed_times).isoformat() if parsed_times else None,
        "sample_unparsed_timestamp_count": len(unparsed_times),
        "sample_negative_sentinel_values": sentinel_examples,
        "raw_examples_by_observed_label": examples,
    },
    "iana": iana_summary,
}
summary_file = inspection_folder / "inspection_summary.json"
summary_file.write_text(json.dumps(inspection_summary, indent=2), encoding="utf-8")
with (inspection_folder / "traffic_headers.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(header_summary[0]))
    writer.writeheader()
    writer.writerows(header_summary)
print("Saved:", summary_file)
print("Saved:", inspection_folder / "traffic_headers.csv")

Saved: /home/jovyan/work/reports/inspection/original/inspection_summary.json
Saved: /home/jovyan/work/reports/inspection/original/traffic_headers.csv


Step 1 is complete. Next, we define the schemas.


## Step 2 - Define the schemas

A schema tells Spark the name and type of each column. We write the types ourselves:

- `StringType`: labels, dates before cleaning, and IP addresses.
- `IntegerType`: ports and protocol codes.
- `LongType`: whole-number counts, flags, header lengths, and flow durations.
- `DoubleType`: rates, averages, ratios, and values with decimals or scientific notation.

For example, some byte totals are written as `0.0`, so they need `DoubleType`. `True` allows missing values, which we will check during cleaning. Dates stay as text until then.

We define one schema with 80 columns and another with the four extra columns.


In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

TRAFFIC_SCHEMA = StructType([
    StructField("Dst Port", IntegerType(), True),
    StructField("Protocol", IntegerType(), True),
    StructField("Timestamp", StringType(), True),
    StructField("Flow Duration", LongType(), True),
    StructField("Tot Fwd Pkts", LongType(), True),
    StructField("Tot Bwd Pkts", LongType(), True),
    StructField("TotLen Fwd Pkts", DoubleType(), True),
    StructField("TotLen Bwd Pkts", DoubleType(), True),
    StructField("Fwd Pkt Len Max", DoubleType(), True),
    StructField("Fwd Pkt Len Min", DoubleType(), True),
    StructField("Fwd Pkt Len Mean", DoubleType(), True),
    StructField("Fwd Pkt Len Std", DoubleType(), True),
    StructField("Bwd Pkt Len Max", DoubleType(), True),
    StructField("Bwd Pkt Len Min", DoubleType(), True),
    StructField("Bwd Pkt Len Mean", DoubleType(), True),
    StructField("Bwd Pkt Len Std", DoubleType(), True),
    StructField("Flow Byts/s", DoubleType(), True),
    StructField("Flow Pkts/s", DoubleType(), True),
    StructField("Flow IAT Mean", DoubleType(), True),
    StructField("Flow IAT Std", DoubleType(), True),
    StructField("Flow IAT Max", DoubleType(), True),
    StructField("Flow IAT Min", DoubleType(), True),
    StructField("Fwd IAT Tot", DoubleType(), True),
    StructField("Fwd IAT Mean", DoubleType(), True),
    StructField("Fwd IAT Std", DoubleType(), True),
    StructField("Fwd IAT Max", DoubleType(), True),
    StructField("Fwd IAT Min", DoubleType(), True),
    StructField("Bwd IAT Tot", DoubleType(), True),
    StructField("Bwd IAT Mean", DoubleType(), True),
    StructField("Bwd IAT Std", DoubleType(), True),
    StructField("Bwd IAT Max", DoubleType(), True),
    StructField("Bwd IAT Min", DoubleType(), True),
    StructField("Fwd PSH Flags", LongType(), True),
    StructField("Bwd PSH Flags", LongType(), True),
    StructField("Fwd URG Flags", LongType(), True),
    StructField("Bwd URG Flags", LongType(), True),
    StructField("Fwd Header Len", LongType(), True),
    StructField("Bwd Header Len", LongType(), True),
    StructField("Fwd Pkts/s", DoubleType(), True),
    StructField("Bwd Pkts/s", DoubleType(), True),
    StructField("Pkt Len Min", DoubleType(), True),
    StructField("Pkt Len Max", DoubleType(), True),
    StructField("Pkt Len Mean", DoubleType(), True),
    StructField("Pkt Len Std", DoubleType(), True),
    StructField("Pkt Len Var", DoubleType(), True),
    StructField("FIN Flag Cnt", LongType(), True),
    StructField("SYN Flag Cnt", LongType(), True),
    StructField("RST Flag Cnt", LongType(), True),
    StructField("PSH Flag Cnt", LongType(), True),
    StructField("ACK Flag Cnt", LongType(), True),
    StructField("URG Flag Cnt", LongType(), True),
    StructField("CWE Flag Count", LongType(), True),
    StructField("ECE Flag Cnt", LongType(), True),
    StructField("Down/Up Ratio", DoubleType(), True),
    StructField("Pkt Size Avg", DoubleType(), True),
    StructField("Fwd Seg Size Avg", DoubleType(), True),
    StructField("Bwd Seg Size Avg", DoubleType(), True),
    StructField("Fwd Byts/b Avg", DoubleType(), True),
    StructField("Fwd Pkts/b Avg", DoubleType(), True),
    StructField("Fwd Blk Rate Avg", DoubleType(), True),
    StructField("Bwd Byts/b Avg", DoubleType(), True),
    StructField("Bwd Pkts/b Avg", DoubleType(), True),
    StructField("Bwd Blk Rate Avg", DoubleType(), True),
    StructField("Subflow Fwd Pkts", LongType(), True),
    StructField("Subflow Fwd Byts", LongType(), True),
    StructField("Subflow Bwd Pkts", LongType(), True),
    StructField("Subflow Bwd Byts", LongType(), True),
    StructField("Init Fwd Win Byts", LongType(), True),
    StructField("Init Bwd Win Byts", LongType(), True),
    StructField("Fwd Act Data Pkts", LongType(), True),
    StructField("Fwd Seg Size Min", LongType(), True),
    StructField("Active Mean", DoubleType(), True),
    StructField("Active Std", DoubleType(), True),
    StructField("Active Max", DoubleType(), True),
    StructField("Active Min", DoubleType(), True),
    StructField("Idle Mean", DoubleType(), True),
    StructField("Idle Std", DoubleType(), True),
    StructField("Idle Max", DoubleType(), True),
    StructField("Idle Min", DoubleType(), True),
    StructField("Label", StringType(), True),
])

TRAFFIC_WITH_IP_SCHEMA = StructType([
    StructField("Flow ID", StringType(), True),
    StructField("Src IP", StringType(), True),
    StructField("Src Port", IntegerType(), True),
    StructField("Dst IP", StringType(), True),
] + TRAFFIC_SCHEMA.fields)

print("Original traffic schemas:", len(TRAFFIC_SCHEMA), "and", len(TRAFFIC_WITH_IP_SCHEMA), "columns")


Original traffic schemas: 80 and 84 columns


### IANA schema

We read IANA columns as text. A port can be a range such as `225-241`, so we cannot convert every value to an integer yet.


In [9]:
IANA_SCHEMA = StructType([
    StructField("Service Name", StringType(), True),
    StructField("Port Number", StringType(), True),
    StructField("Transport Protocol", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Assignee", StringType(), True),
    StructField("Contact", StringType(), True),
    StructField("Registration Date", StringType(), True),
    StructField("Modification Date", StringType(), True),
    StructField("Reference", StringType(), True),
    StructField("Service Code", StringType(), True),
    StructField("Unauthorized Use Reported", StringType(), True),
    StructField("Assignment Notes", StringType(), True),
])

print("IANA schema:", len(IANA_SCHEMA), "columns")

IANA schema: 12 columns


### Check each header against its schema

The column names and order must match. If they do not, stop and check the file. Step 3 will test whether the values can be read with these types.


In [10]:
traffic_schemas = {}
for file_name, header in traffic_headers.items():
    schema = TRAFFIC_WITH_IP_SCHEMA if "Flow ID" in header else TRAFFIC_SCHEMA
    assert header == schema.fieldNames(), f"Check the columns in {file_name}"
    traffic_schemas[file_name] = schema
    print(f"Header matches: {file_name} ({len(schema)} columns)")

assert registry_header == IANA_SCHEMA.fieldNames(), "Check the IANA columns"
print("IANA header matches.")


Header matches: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv (84 columns)
Header matches: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
IANA header matches.


Step 2 is complete. Next, we test the schemas with the samples.


## Step 3 - Test CSV to Parquet

We use the 5,000 traffic rows from Step 1 and the whole IANA file to test reading CSV, writing Parquet, and reading it back. This checks the code before we process all the traffic files.


### 3A - Save the samples and start Spark

Save one small CSV per traffic file in `data/intermediate/original/`. Each keeps its own header. Start Spark with two local worker threads and four shuffle partitions.


In [11]:
sample_folder = ROOT / "data" / "intermediate" / "original"
sample_folder.mkdir(parents=True, exist_ok=True)
for path in traffic_files:
    with (sample_folder / path.name).open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=traffic_headers[path.name])
        writer.writeheader()
        writer.writerows(sample_rows_by_file[path.name])

from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]")
         .appName("DPA-Volume-Original-Sample")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Sample traffic records:", len(sample_rows))


Sample traffic records: 5000


### 3B - Read the CSVs

Read each file with its schema. `FAILFAST` stops if a value cannot be parsed. IANA needs `multiLine=True` because some quoted fields contain line breaks.

`unionByName` combines files using column names. `allowMissingColumns=True` fills the four extra columns with null where a file does not have them. These are not recovered IP addresses.

Use lowercase column names with underscores to make them easier to use.


In [12]:
traffic_frames = []
for path in traffic_files:
    frame = (spark.read.schema(traffic_schemas[path.name])
             .option("header", True).option("enforceSchema", False)
             .option("mode", "FAILFAST").option("escape", '"')
             .csv(str(sample_folder / path.name)))
    traffic_frames.append(frame)

traffic_df = traffic_frames[0]
for frame in traffic_frames[1:]:
    traffic_df = traffic_df.unionByName(frame, allowMissingColumns=True)

iana_df = (spark.read.schema(IANA_SCHEMA)
           .option("header", True).option("enforceSchema", False)
           .option("mode", "FAILFAST").option("escape", '"')
           .option("multiLine", True).csv(str(registry_file)))

traffic_df = traffic_df.toDF(*[name.lower().replace(" ", "_").replace("/", "_") for name in traffic_df.columns])
iana_df = iana_df.toDF(*[name.lower().replace(" ", "_") for name in iana_df.columns])

traffic_df.select("dst_port", "protocol", "flow_duration", "label").printSchema()
traffic_df.select("dst_port", "protocol", "flow_duration", "label").show(5)


root
 |-- dst_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- label: string (nullable = true)



+--------+--------+-------------+------+
|dst_port|protocol|flow_duration| label|
+--------+--------+-------------+------+
|     443|       6|       141385|Benign|
|   49684|       6|          281|Benign|
|     443|       6|       279824|Benign|
|     443|       6|          132|Benign|
|     443|       6|       274016|Benign|
+--------+--------+-------------+------+
only showing top 5 rows


### 3C - Save and check Parquet

Parquet keeps the column types. Save the test tables in `data/parquet/original/sample/`, then read them back. Check the row counts, types, and contents. `exceptAll` compares rows while keeping duplicate counts.


In [13]:
parquet_folder = ROOT / "data" / "parquet" / "original" / "sample"
traffic_df.write.mode("overwrite").parquet(str(parquet_folder / "traffic"))
iana_df.write.mode("overwrite").parquet(str(parquet_folder / "iana"))

traffic_parquet = spark.read.parquet(str(parquet_folder / "traffic"))
iana_parquet = spark.read.parquet(str(parquet_folder / "iana"))

for name, original, restored, expected in [
    ("Traffic", traffic_df, traffic_parquet, len(sample_rows)),
    ("IANA", iana_df, iana_parquet, len(registry_rows)),
]:
    actual = restored.count()
    assert actual == expected, f"{name}: row count changed"
    assert original.dtypes == restored.dtypes, f"{name}: column types changed"
    assert original.exceptAll(restored).count() == 0, f"{name}: row contents changed"
    print(f"{name}: {actual} rows; types and contents preserved.")

print("Step 3 passed. Parquet saved in:", parquet_folder)

Traffic: 5000 rows; types and contents preserved.


IANA: 14536 rows; types and contents preserved.
Step 3 passed. Parquet saved in: /home/jovyan/work/data/parquet/original/sample


Step 3 is complete. The sample tables passed the checks. Step 4 uses all original traffic rows, not these samples.


## Step 4 - Read and clean all the data

We read all ten original traffic files and the whole IANA file. We save typed Parquet before cleaning, then work from those tables. We leave the source CSVs unchanged.

Before each cleaning step, we explain why we need it. We record how many rows were removed or changed.

The [paper's abstract](https://www.nu.edu.eg/publications/apache-spark-powered-enhancing-network-intrusion-detection-system-using-random-forest) also mentions missing values, duplicates, and unused columns. We could not access the full methods. The rules below are our choices for our research questions.


### 4A - Read all files and save raw Parquet

**Why:** The sample only tested the code. Now we need all the data, using the correct schema for each file.

`PERMISSIVE` puts badly formatted records in `_corrupt_record`. We save them in raw Parquet so we can count them before removal. We disable CSV column pruning so Spark checks every field. `source_file` keeps the original filename.

The missing IP columns in nine files are expected. We will not remove rows because of them.


In [14]:
from pyspark.sql import functions as F
from pyspark import StorageLevel

files_to_clean = traffic_files  # All original CSVs, including 20 February.
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "16")
full_frames = []
for path in files_to_clean:
    schema = StructType(traffic_schemas[path.name].fields + [StructField("_corrupt_record", StringType(), True)])
    frame = (spark.read.schema(schema)
             .option("header", True).option("enforceSchema", False)
             .option("mode", "PERMISSIVE").option("escape", '"')
             .option("columnNameOfCorruptRecord", "_corrupt_record")
             .csv(str(path)).withColumn("source_file", F.lit(path.name)))
    full_frames.append(frame)
traffic_full = full_frames[0]
for frame in full_frames[1:]:
    traffic_full = traffic_full.unionByName(frame, allowMissingColumns=True)
traffic_full = traffic_full.toDF(*[c.lower().replace(" ", "_").replace("/", "_") for c in traffic_full.columns])

raw_folder = ROOT / "data" / "parquet" / "original" / "raw"
traffic_full.write.mode("overwrite").parquet(str(raw_folder / "traffic"))
iana_df.write.mode("overwrite").parquet(str(raw_folder / "iana"))
traffic_raw = spark.read.parquet(str(raw_folder / "traffic"))
iana_raw = spark.read.parquet(str(raw_folder / "iana"))
print("Original files processed:", len(files_to_clean))
traffic_raw.groupBy("source_file").count().orderBy("source_file").show(truncate=False)


Original files processed: 10


+--------------------------------------------------+-------+
|source_file                                       |count  |
+--------------------------------------------------+-------+
|Friday-02-03-2018_TrafficForML_CICFlowMeter.csv   |1048575|
|Friday-16-02-2018_TrafficForML_CICFlowMeter.csv   |1048575|
|Friday-23-02-2018_TrafficForML_CICFlowMeter.csv   |1048575|
|Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv |7948748|
|Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv |331125 |
|Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv |1048575|
|Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv |1048575|
|Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv|1048575|
|Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv|1048575|
|Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv|613104 |
+--------------------------------------------------+-------+



### Keep a record of each step

For each rule, save the input, removed, changed, and remaining row counts, with a reason. Each removal count starts from the previous step, so a row removed earlier is not counted again.

Changing values or types counts as a modification. Selecting fewer columns does not. Steps that only change columns keep the same row count, so we do not need to count all the rows again.


In [3]:
audit_rows = []

def record_step(dataset, step, before, after, reason, modified=0):
    audit_rows.append((dataset, step, before, before - after, modified, after, reason))
    print(f"{dataset} | {step}: {before:,} -> {after:,}; removed {before - after:,}; modified {modified:,}")
    return after

traffic_clean = traffic_raw
iana_clean = iana_raw
traffic_count = traffic_clean.count()
iana_count = iana_clean.count()
print(f"Input rows: traffic={traffic_count:,}, IANA={iana_count:,}")


Input rows: traffic=16,233,002, IANA=14,536


### 4B - Remove badly formatted records

**Why:** Rows with the wrong number of fields or invalid numeric text do not match our schema. A repeated header inside a file is one example. We remove these rows from the working table. Their original text stays in raw Parquet.


In [4]:
traffic_clean = traffic_clean.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
traffic_count = record_step("traffic", "Malformed records", traffic_count, traffic_clean.count(),
                            "Remove CSV structure or numeric parsing failures; raw evidence retained")


traffic | Malformed records: 16,233,002 -> 16,232,943; removed 59; modified 0


### 4C - Remove duplicate records

**Why:** Repeated rows would be counted more than once in the analysis. We compare all source columns before removing any columns. For traffic, `source_file` means we check duplicates within each file.

These are matching CSV records. Nine files have no host IPs, so we cannot prove they are the same network event. For IANA, we also compare the complete rows first.


In [5]:
traffic_clean = traffic_clean.dropDuplicates().persist(StorageLevel.DISK_ONLY)
traffic_count = record_step("traffic", "Duplicate records", traffic_count, traffic_clean.count(),
                            "Keep one matching typed source record per file, comparing all source columns")
iana_clean = iana_clean.dropDuplicates()
iana_count = record_step("iana", "Duplicate records", iana_count, iana_clean.count(),
                         "Keep one copy of each complete registry record")


traffic | Duplicate records: 16,232,943 -> 15,822,236; removed 410,707; modified 0


iana | Duplicate records: 14,536 -> 14,536; removed 0; modified 0


### 4D - Keep the columns we need

**Why:** We need labels, dates, durations, packets, bytes, and rates to compare traffic. Destination port and protocol are needed for the service lookup. We also keep the filename.

For IANA, we need service name, port number, and transport protocol. We drop the other columns from the working tables. Raw Parquet still keeps every source column.

We save reused traffic tables on disk with `persist` to avoid repeating the duplicate check and to limit memory use. We release them with `unpersist` when finished.


In [6]:
analysis_columns = ["dst_port", "protocol", "timestamp", "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",
                    "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_byts_s", "flow_pkts_s", "label"]
wide_traffic = traffic_clean
traffic_base = wide_traffic.select(*analysis_columns, "source_file").persist(StorageLevel.DISK_ONLY)
traffic_clean = traffic_base
traffic_count = record_step("traffic", "Select analysis columns", traffic_count, traffic_clean.count(),
                            "Keep 11 research fields and source_file; raw table retains all columns")
wide_traffic.unpersist()
iana_clean = iana_clean.select("service_name", "port_number", "transport_protocol")
iana_count = record_step("iana", "Select lookup columns", iana_count, iana_count,
                         "Keep service name, port number, and transport protocol")


traffic | Select analysis columns: 15,822,236 -> 15,822,236; removed 0; modified 0
iana | Select lookup columns: 14,536 -> 14,536; removed 0; modified 0


### 4E - Fix spaces and empty text

**Why:** Extra spaces can split matching values into different groups. We trim labels, dates, and IANA fields. We lowercase IANA protocols and turn empty text into null. We keep the original attack-label spelling.


In [7]:
traffic_modified = traffic_clean.filter(
    (F.col("label") != F.trim("label")) | (F.trim("label") == "") |
    (F.col("timestamp") != F.trim("timestamp")) | (F.trim("timestamp") == "")).count()
for c in ["label", "timestamp"]:
    traffic_clean = traffic_clean.withColumn(c, F.when(F.trim(c) != "", F.trim(c)))
traffic_count = record_step("traffic", "Normalize text", traffic_count, traffic_count,
                            "Trim labels/timestamps and turn blank text into null", traffic_modified)

iana_modified = iana_clean.filter(
    (F.col("service_name") != F.trim("service_name")) | (F.trim("service_name") == "") |
    (F.col("port_number") != F.trim("port_number")) | (F.trim("port_number") == "") |
    (F.col("transport_protocol") != F.lower(F.trim("transport_protocol"))) |
    (F.trim("transport_protocol") == "")).count()
for c in iana_clean.columns:
    iana_clean = iana_clean.withColumn(c, F.when(F.trim(c) != "", F.trim(c)))
iana_clean = iana_clean.withColumn("transport_protocol", F.lower("transport_protocol"))
iana_count = record_step("iana", "Normalize text", iana_count, iana_count,
                         "Trim lookup fields, lowercase protocols, and turn blank text into null", iana_modified)


traffic | Normalize text: 15,822,236 -> 15,822,236; removed 0; modified 0


iana | Normalize text: 14,536 -> 14,536; removed 0; modified 0


### 4F - Check and remove missing values

**Why:** Our comparisons need all 11 selected traffic fields. We print the missing counts, then drop rows with null or `NaN` in these fields. `NaN` means a number is undefined. Filling in labels, dates, or traffic rates would add guesses to our results.

The IANA lookup needs a service name, port, and protocol. We exclude rows missing these fields from the lookup, even though they may be valid registry entries. Missing IP columns are not part of this check.


In [8]:
numeric_columns = [c for c in analysis_columns if c not in ["label", "timestamp"]]
missing_checks = []
for c in analysis_columns:
    missing = F.col(c).isNull()
    if c in numeric_columns:
        missing = missing | F.isnan(c)
    missing_checks.append(F.sum(F.when(missing, 1).otherwise(0)).alias(c))
missing_counts = traffic_clean.select(*missing_checks).first().asDict()
print("Traffic missing values before this rule:", {c: n for c, n in missing_counts.items() if n})
print("IANA missing values before this rule:", iana_clean.select(*[
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in iana_clean.columns]).first().asDict())
traffic_clean = traffic_clean.na.drop(subset=analysis_columns)
traffic_count = record_step("traffic", "Missing required values", traffic_count, traffic_clean.count(),
                            "Drop null/NaN in required research fields; do not impute traffic or labels")
iana_clean = iana_clean.na.drop(subset=["service_name", "port_number", "transport_protocol"])
iana_count = record_step("iana", "Missing lookup values", iana_count, iana_clean.count(),
                         "A named-service lookup requires service, port, and transport protocol")


Traffic missing values before this rule: {'flow_byts_s': 58343}


IANA missing values before this rule: {'service_name': 1724, 'port_number': 1073, 'transport_protocol': 1467}


traffic | Missing required values: 15,822,236 -> 15,763,893; removed 58,343; modified 0


iana | Missing lookup values: 14,536 -> 11,723; removed 2,813; modified 0


### 4G - Remove infinite values

**Why:** An infinite rate cannot be used in our comparisons. We remove rows with infinity in any selected numeric field. Missing values were removed first, so a row with both problems is counted only once.


In [9]:
finite = F.lit(True)
for c in numeric_columns:
    finite = finite & (F.abs(F.col(c).cast("double")) != F.lit(float("inf")))
traffic_clean = traffic_clean.filter(finite)
traffic_count = record_step("traffic", "Infinite measurements", traffic_count, traffic_clean.count(),
                            "Retained numeric measurements must be finite")


traffic | Infinite measurements: 15,763,893 -> 15,727,888; removed 36,005; modified 0


### 4H - Check impossible values

**Why:** Durations, packet counts, bytes, and rates cannot be negative. Ports must be from 0 to 65535, and protocol codes from 0 to 255. Positive bytes with zero packets in the same direction are also inconsistent.

We keep zero ports, zero durations, and protocols other than TCP/UDP. Large finite values can be part of an attack, so we do not remove them just because they are high. The unused TCP window columns can contain special negative values such as `-1`.


In [10]:
valid = F.col("dst_port").between(0, 65535) & F.col("protocol").between(0, 255)
for c in ["flow_duration", "tot_fwd_pkts", "tot_bwd_pkts", "totlen_fwd_pkts", "totlen_bwd_pkts",
          "flow_byts_s", "flow_pkts_s"]:
    valid = valid & (F.col(c) >= 0)
valid = valid & ((F.col("tot_fwd_pkts") > 0) | (F.col("totlen_fwd_pkts") == 0))
valid = valid & ((F.col("tot_bwd_pkts") > 0) | (F.col("totlen_bwd_pkts") == 0))
traffic_clean = traffic_clean.filter(valid)
traffic_count = record_step("traffic", "Impossible measurements or codes", traffic_count, traffic_clean.count(),
                            "Reject negative measurements, invalid port/protocol ranges, and bytes without packets")


traffic | Impossible measurements or codes: 15,727,888 -> 15,727,874; removed 14; modified 0


### 4I - Convert and check dates

**Why:** We need timestamp values for time comparisons. We read day/month/year, including fractional seconds. We remove invalid dates and dates outside 14 February to 2 March 2018.

We check the whole collection period because traffic can cross midnight. The source gives no timezone offset, so we keep its clock times. Every retained text-to-timestamp conversion counts as a changed row.


In [11]:
traffic_clean = traffic_clean.withColumn("timestamp", F.coalesce(
    F.try_to_timestamp("timestamp", F.lit("dd/MM/yyyy HH:mm:ss")),
    F.try_to_timestamp("timestamp", F.lit("dd/MM/yyyy HH:mm:ss.SSSSSS"))))
traffic_clean = traffic_clean.filter(
    F.col("timestamp").isNotNull() &
    (F.col("timestamp") >= F.lit("2018-02-14").cast("timestamp")) &
    (F.col("timestamp") < F.lit("2018-03-03").cast("timestamp")))
parsed_count = traffic_clean.count()
traffic_count = record_step("traffic", "Parse and validate timestamps", traffic_count, parsed_count,
                            "Convert valid timestamps; reject invalid dates outside the collection period", parsed_count)


traffic | Parse and validate timestamps: 15,727,874 -> 15,727,874; removed 0; modified 15,727,874


### 4J - Prepare individual IANA ports

**Why:** Each traffic row has one destination port. For a simple join, we keep individual TCP/UDP ports, convert them to integers, and check the range 0 to 65535.

Port ranges and other protocols are valid IANA entries, but this lookup does not use them. Later, an unmatched flow will not necessarily mean an unregistered port.


In [12]:
iana_clean = iana_clean.filter(
    F.col("port_number").rlike(r"^[0-9]{1,5}$") & F.col("transport_protocol").isin("tcp", "udp"))
iana_clean = iana_clean.withColumn("port_number", F.col("port_number").cast("int"))
iana_clean = iana_clean.filter(F.col("port_number").between(0, 65535))
port_count = iana_clean.count()
iana_count = record_step("iana", "Individual TCP/UDP ports", iana_count, port_count,
                         "Lookup scope: individual TCP/UDP registrations; convert retained ports to integers", port_count)


iana | Individual TCP/UDP ports: 11,723 -> 11,623; removed 100; modified 11,623


### 4K - Combine service names sharing a port

**Why:** Several service names can share a port and protocol. Joining all of them separately would multiply traffic rows. We keep the names together in one sorted string per port and protocol. These aliases are valid names, not duplicate errors.


In [13]:
alias_groups = iana_clean.groupBy("port_number", "transport_protocol").agg(
    F.sort_array(F.collect_set("service_name")).alias("service_names"))
alias_modified = alias_groups.filter(F.size("service_names") > 1).count()
iana_clean = alias_groups.select("port_number", "transport_protocol",
                                 F.concat_ws(", ", "service_names").alias("service_name"))
iana_count = record_step("iana", "Combine service aliases", iana_count, iana_clean.count(),
                         "Aggregate all distinct service names into one row per port/protocol; prevent join multiplication", alias_modified)


iana | Combine service aliases: 11,623 -> 11,391; removed 232; modified 226


### 4L - Save the results

Save the cleaned tables as Parquet. Read them back and check counts and types. Save the cleaning counts and reasons in one CSV. Rerunning replaces these generated files.


In [14]:
clean_folder = ROOT / "data" / "parquet" / "original" / "cleaned"
traffic_clean.write.mode("overwrite").parquet(str(clean_folder / "traffic"))
iana_clean.write.mode("overwrite").parquet(str(clean_folder / "iana"))
clean_traffic = spark.read.parquet(str(clean_folder / "traffic"))
clean_iana = spark.read.parquet(str(clean_folder / "iana"))
assert clean_traffic.count() == traffic_count
assert clean_iana.count() == iana_count
assert clean_traffic.dtypes == traffic_clean.dtypes
assert clean_iana.dtypes == iana_clean.dtypes
traffic_base.unpersist()

audit_columns = ["dataset", "step", "input_rows", "removed_rows", "modified_rows", "output_rows", "reason"]
cleaning_audit = spark.createDataFrame(audit_rows, audit_columns)
cleaning_audit.show(truncate=False)
audit_folder = ROOT / "reports" / "cleaning" / "original"
audit_folder.mkdir(parents=True, exist_ok=True)
with (audit_folder / "cleaning_audit.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.writer(output)
    writer.writerow(audit_columns)
    writer.writerows(audit_rows)
print("Cleaned Parquet:", clean_folder)
print("Cleaning audit:", audit_folder / "cleaning_audit.csv")


+-------+--------------------------------+----------+------------+-------------+-----------+------------------------------------------------------------------------------------------------+
|dataset|step                            |input_rows|removed_rows|modified_rows|output_rows|reason                                                                                          |
+-------+--------------------------------+----------+------------+-------------+-----------+------------------------------------------------------------------------------------------------+
|traffic|Malformed records               |16233002  |59          |0            |16232943   |Remove CSV structure or numeric parsing failures; raw evidence retained                         |
|traffic|Duplicate records               |16232943  |410707      |0            |15822236   |Keep one matching typed source record per file, comparing all source columns                    |
|iana   |Duplicate records               |14536   

After Step 4 runs, `clean_traffic` and `clean_iana` are ready for IANA enrichment. The audit gives the actual row counts. The join, research analysis, charts, and timing comparison are still to do.
